In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install transformers datasets evaluate accelerate
!pip install scikit-learn pandas numpy
!pip install jiwer  # WER metric backend
!pip install huggingface_hub

Looking in indexes: https://download.pytorch.org/whl/cu118
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 110.1 MB/s eta 0:00:00


In [ ]:
!git clone https://github.com/unza-speech-lab/zambezi-voice-nyanja.git


fatal: destination path 'zambezi-voice-nyanja' already exists and is not an empty directory.


In [ ]:
# ============================================================
# ASR — Nyanja  |  Whisper Medium (~244 M parameters)
# ============================================================


# ============================================================
# 1  Imports
# ============================================================

import os
import re
import torch
import pandas as pd
import numpy as np
from dataclasses import dataclass
from typing import Any, Dict, List

from datasets import Dataset, Audio
from evaluate import load
from sklearn.model_selection import train_test_split
from transformers import (
    WhisperFeatureExtractor,
    WhisperTokenizer,
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    set_seed,
)
import transformers

print("Transformers version:", transformers.__version__)
set_seed(42)

# ============================================================
# 2  Paths
# ============================================================

DATA_DIR   = "/content/zambezi-voice-nyanja/nya/"
AUDIO_DIR  = os.path.join(DATA_DIR, "audio/")
TRAIN_TSV  = os.path.join("/content/zambezi-voice-nyanja/nya/train.tsv")
OUTPUT_DIR = "./nyanja-whisper-model"
FINAL_DIR  = "./nyanja-whisper-final"
MODEL_NAME = "openai/whisper-small"

# ============================================================
# 3  Load Dataset
# ============================================================

print("Loading Nyanja dataset...")

df = pd.read_csv(TRAIN_TSV, sep="\t")
df = df[["audio_id", "sentence"]].rename(columns={"sentence": "text"})
df["path"] = AUDIO_DIR + df["audio_id"]
df = df.dropna()

total_before = len(df)
df["exists"]  = df["path"].apply(os.path.exists)
found_df      = df[df["exists"]]
missing_df    = df[~df["exists"]]

print(f"Total rows in TSV (after dropna): {total_before}")
print(f"Audio files FOUND:   {len(found_df)}")
print(f"Audio files MISSING: {len(missing_df)}")

if len(missing_df) > 0:
    print(f"Missing rate: {len(missing_df)/total_before*100:.1f}%")
    for _, row in missing_df.head(10).iterrows():
        print(f"  {row['path']}")

df = df[df["exists"]][["path", "text"]].reset_index(drop=True)
print(f"\nUsable samples: {len(df)}")

# ============================================================
# 4  Train / Validation Split
# ============================================================

train_df, val_df = train_test_split(df, test_size=0.1, random_state=42)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

print("Train:", len(train_df))
print("Validation:", len(val_df))

# ============================================================
# 5  HuggingFace Dataset
# ============================================================

def create_dataset(df):
    dataset = Dataset.from_pandas(df)
    dataset = dataset.cast_column("path", Audio(sampling_rate=16000))
    return dataset

train_dataset = create_dataset(train_df)
val_dataset   = create_dataset(val_df)

# ============================================================
# 6  Clean Text  (Nyanja-specific)
# ============================================================

CHARS_TO_REMOVE    = r'[,?.!\-;:\"\"%\'\"\'\u2018\u2019\u2026()\[\]{}]'
NON_NYANJA_LETTERS = r'[qx]'

def clean_text(batch):
    text = batch["text"].lower()
    text = re.sub(CHARS_TO_REMOVE, "", text)
    text = re.sub(r"[0-9]", "", text)
    text = re.sub(NON_NYANJA_LETTERS, "", text)
    text = re.sub(r"[^\x00-\x7F]", "", text)   # strip non-ASCII
    text = re.sub(r" +", " ", text).strip()
    batch["text"] = text
    return batch

train_dataset = train_dataset.map(clean_text)
val_dataset   = val_dataset.map(clean_text)

train_dataset = train_dataset.filter(lambda x: len(x["text"].strip()) > 0)
val_dataset   = val_dataset.filter(lambda x: len(x["text"].strip()) > 0)

print("\n--- Sample cleaned text ---")
for i in range(5):
    print(repr(train_dataset[i]["text"]))
print("---\n")

# ============================================================
# Whisper processor  — Nyanja (unsupported language workaround)
# ============================================================
feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_NAME)

# Don't pass language= here; Nyanja is not in Whisper's vocab
tokenizer = WhisperTokenizer.from_pretrained(MODEL_NAME, task="transcribe")
processor  = WhisperProcessor(feature_extractor=feature_extractor, tokenizer=tokenizer)
# ============================================================
# 8  Prepare Dataset
#    Whisper expects 80-bin log-mel spectrograms, not raw
#    waveforms. The feature extractor handles this automatically.
# ============================================================

def prepare_dataset(batch):
    audio = batch["path"]
    batch["input_features"] = feature_extractor(
        audio["array"],
        sampling_rate=audio["sampling_rate"],
        return_tensors="pt",
    ).input_features[0]
    batch["labels"] = tokenizer(batch["text"]).input_ids
    return batch

train_dataset = train_dataset.map(
    prepare_dataset,
    remove_columns=train_dataset.column_names,
    num_proc=1,
)
val_dataset = val_dataset.map(
    prepare_dataset,
    remove_columns=val_dataset.column_names,
    num_proc=1,
)

print("Train after preparation:", len(train_dataset))
print("Validation after preparation:", len(val_dataset))

# ============================================================
# 9  Data Collator  (Seq2Seq — not CTC)
#    - Pads log-mel spectrograms to the same length
#    - Pads label sequences and masks padding with -100
#      so the cross-entropy loss ignores them
# ============================================================

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        # Pad input features (log-mel spectrograms)
        input_features = [
            {"input_features": f["input_features"]} for f in features
        ]
        batch = self.processor.feature_extractor.pad(
            input_features, return_tensors="pt"
        )

        # Pad label sequences
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(
            label_features, return_tensors="pt"
        )

        # Replace padding token id with -100 (ignored in loss)
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )

        # Remove BOS token if prepended by the tokenizer
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

# ============================================================
# 10  Load Whisper Medium
#
#     openai/whisper-medium
#     - ~244 M parameters  (closest to 255 M in Whisper family)
#     - Encoder-decoder transformer pre-trained on 680 k hours
#       of multilingual audio covering 99 languages
#     - Full model fine-tuned on Nyanja
#
#     Other Whisper options to experiment with:
#       openai/whisper-small      (~244 M, faster)
#       openai/whisper-large-v3   (~1.5 B, best quality, slow)
# ============================================================

# ============================================================
# Load model
# ============================================================
model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)

# ✅ FIXED for Transformers 5.0 — use generation_config, not model.config
model.generation_config.forced_decoder_ids = None
model.generation_config.suppress_tokens     = []
model.generation_config.language            = None
model.generation_config.task                = "transcribe"

print(f"Model: {MODEL_NAME}  |  Parameters: {model.num_parameters():,}")
# ============================================================
# Training arguments
# ============================================================
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    eval_strategy="steps",
    eval_steps=200,
    logging_steps=100,
    save_steps=200,
    save_strategy="steps",
    num_train_epochs=5,
    learning_rate=1e-5,
    warmup_steps=500,
    weight_decay=0.01,
    gradient_checkpointing=True,
    fp16=True,
    predict_with_generate=True,
    generation_max_length=225,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    report_to="none",
)


Transformers version: 5.0.0
Loading Nyanja dataset...
Total rows in TSV (after dropna): 8117
Audio files FOUND:   8117
Audio files MISSING: 0

Usable samples: 8117
Train: 7305
Validation: 812


Map:   0%|          | 0/7305 [00:00<?, ? examples/s]

Map:   0%|          | 0/812 [00:00<?, ? examples/s]

Filter:   0%|          | 0/7305 [00:00<?, ? examples/s]

Filter:   0%|          | 0/812 [00:00<?, ? examples/s]

...

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

Model: openai/whisper-small  |  Parameters: 241,734,912


In [ ]:
# ============================================================
# 11  FIX: Remove legacy generation parameters from model config
# ============================================================
model.config.suppress_tokens = None   # <-- ADD THIS LINE

# ============================================================
# 12  WER Metric
# ============================================================
wer_metric = load("wer")

def compute_metrics(pred):
    pred_ids  = pred.predictions
    label_ids = pred.label_ids

    # Replace -100 padding before decoding
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str  = processor.tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

# ============================================================
# 13  Trainer
# ============================================================
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=processor,
    compute_metrics=compute_metrics,
)

# ============================================================
# 14  Train
# ============================================================
print("\n========== STARTING TRAINING ==========")
print(f"Model                : {MODEL_NAME}")
print(f"Transformers version : {transformers.__version__}")
print(f"Train samples        : {len(train_dataset)}")
print(f"Val samples          : {len(val_dataset)}")
print(f"Epochs               : {training_args.num_train_epochs}")
print(f"Effective batch size : {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"Learning rate        : {training_args.learning_rate}")
print("========================================\n")

trainer.train()

# ============================================================
# 15  Save Model & Processor
# ============================================================
model.save_pretrained(FINAL_DIR)
processor.save_pretrained(FINAL_DIR)
print(f"Model and processor saved to {FINAL_DIR}")

# ============================================================
# 16  Final Evaluation
# ============================================================
print("\n========== FINAL EVALUATION ==========")
eval_results = trainer.evaluate()
print(f"Final WER: {eval_results['eval_wer']:.4f}  ({eval_results['eval_wer']*100:.2f}%)")

# ============================================================
# 17  Test Inference  (spot-check 5 validation samples)
# ============================================================
def transcribe(idx):
    example = val_dataset[idx]
    device  = model.device

    input_features = torch.tensor(
        example["input_features"]
    ).unsqueeze(0).to(device)

    with torch.no_grad():
        predicted_ids = model.generate(
            input_features,
            forced_decoder_ids=None,  # ✅ nya not in Whisper vocab
        )

    prediction = processor.tokenizer.decode(
        predicted_ids[0], skip_special_tokens=True
    )

    label_ids = [i for i in example["labels"] if i != -100]
    reference = processor.tokenizer.decode(
        label_ids, skip_special_tokens=True
    )

    print(f"\n--- Sample {idx} ---")
    print("REFERENCE :", reference)
    print("PREDICTION:", prediction)

for i in range(5):
    transcribe(i)

print("\n========== TRAINING PIPELINE COMPLETE ==========")


========== STARTING TRAINING ==========
Model                : openai/whisper-small
Transformers version : 5.0.0
Train samples        : 7305
Val samples          : 812
Epochs               : 5
Effective batch size : 8
Learning rate        : 1e-05



Step,Training Loss,Validation Loss,Wer
200,0.590893,0.440268,0.664442
400,0.748386,0.343601,0.565848
600,0.599644,0.284155,0.257430
800,0.533439,0.253927,0.226731
1000,0.333055,0.231670,0.195408
1200,0.309351,0.221219,0.202972
1400,0.308238,0.211727,0.179302
1600,0.283329,0.203857,0.168268
1800,0.262639,0.201189,0.166400
2000,0.140727,0.199176,0.156345


Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.log

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]